In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import torch
import faiss
import json

from sentence_transformers import SentenceTransformer

print("Python:", sys.version)
print("CUDA:", torch.cuda.is_available())

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
CUDA: True


In [2]:
import ollama

response = ollama.chat(
    model="qwen3:8b",
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: Ollama works"
        }
    ]
)

print(response.message.content)

Ollama works


In [3]:
df = pd.read_csv(
    "../data/processed/m_absa_selected.csv"
)

reviews = (
    df[["text", "language", "domain"]]
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print("Unique reviews:", len(reviews))
display(reviews.head())

,text,language,domain
0,We have gone for dinner only a few times but t...,en,restaurant
1,"Its dark , and cozy . . there is always jazz m...",en,restaurant
2,This place has great indian chinese food .,en,restaurant
3,Not what I would expect for the price and pres...,en,restaurant
4,Finally a reliable Chinese restaurant !,en,restaurant


In [4]:
train_reviews = (
    reviews[reviews["text"].isin(
        df[df["split"] == "train"]["text"].unique()
    )]
    .reset_index(drop=True)
)

print("Train reviews:", len(train_reviews))

Train reviews: 10355


In [5]:
index = faiss.read_index(
    "../vector_store/index.faiss"
)

metadata = pd.read_pickle(
    "../vector_store/metadata.pkl"
)

print("FAISS vectors:", index.ntotal)
print("Metadata rows:", len(metadata))

FAISS vectors: 10355
Metadata rows: 10355


In [6]:
assert index.ntotal == len(metadata)

print("✓ FAISS and metadata are aligned")

✓ FAISS and metadata are aligned


In [7]:
EMBEDDING_MODEL = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", embedding_model.device)

Device: cuda:0


In [8]:
def retrieve_reviews(
    query,
    top_k=5,
    language=None,
    domain=None
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Берём больше результатов, если потом будем фильтровать
    search_k = max(top_k * 5, 20)

    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = metadata.iloc[indices[0]].copy()
    results["score"] = scores[0]

    if language is not None:
        results = results[
            results["language"] == language
        ]

    if domain is not None:
        results = results[
            results["domain"] == domain
        ]

    return (
        results
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [9]:
query = "The hotel room was very clean"

results = retrieve_reviews(
    query,
    top_k=5,
    domain="hotel"
)

display(
    results[
        ["text", "language", "domain", "score"]
    ]
)

,text,language,domain,score
0,Hotel hat viel Charakter und die Zimmer waren ...,de,hotel,0.924704
1,The rooms were clean hotel staff friendly .,en,hotel,0.922473
2,Die Zimmer waren sauber Hotelpersonal freundlich.,de,hotel,0.916726
3,Hotel has alot of charater and the rooms were ...,en,hotel,0.880809
4,"Die Einrichtung war wirklich modern und schön,...",de,hotel,0.876846


In [10]:
def build_context(results):
    context_parts = []

    for i, row in results.iterrows():
        context_parts.append(
            f"[Review {i + 1}]\n"
            f"Language: {row['language']}\n"
            f"Domain: {row['domain']}\n"
            f"Similarity: {row['score']:.3f}\n"
            f"Text: {row['text']}"
        )

    return "\n\n".join(context_parts)

In [11]:
context = build_context(results)

print(context)

[Review 1]
Language: de
Domain: hotel
Similarity: 0.925
Text: Hotel hat viel Charakter und die Zimmer waren sauber.

[Review 2]
Language: en
Domain: hotel
Similarity: 0.922
Text: The rooms were clean  hotel staff friendly .

[Review 3]
Language: de
Domain: hotel
Similarity: 0.917
Text: Die Zimmer waren sauber Hotelpersonal freundlich.

[Review 4]
Language: en
Domain: hotel
Similarity: 0.881
Text: Hotel has alot of charater and the rooms were clean .

[Review 5]
Language: de
Domain: hotel
Similarity: 0.877
Text: Die Einrichtung war wirklich modern und schön, das Hotel war sauber und die Betten waren komfortabel.


In [12]:
def build_rag_prompt(query, context):

    prompt = f"""
You are a review intelligence assistant.

Analyze the user query using ONLY the retrieved customer reviews.

User query:
{query}

Retrieved reviews:
{context}

Return the result ONLY as valid JSON with this structure:

{{
    "main_finding": "short summary of the main finding",
    "common_opinion": "what customers commonly think",
    "negative_signals": [
        "negative or problematic signals"
    ],
    "key_aspects": [
        "important aspects mentioned in the reviews"
    ],
    "conclusion": "short overall conclusion"
}}

Rules:
- Use only information supported by the retrieved reviews.
- Do not invent facts.
- If there are no negative signals, return an empty list.
- Keep the answer concise.
"""

    return prompt.strip()

In [13]:
prompt = build_rag_prompt(
    query,
    context
)

print(prompt)

You are a review intelligence assistant.

Analyze the user query using ONLY the retrieved customer reviews.

User query:
The hotel room was very clean

Retrieved reviews:
[Review 1]
Language: de
Domain: hotel
Similarity: 0.925
Text: Hotel hat viel Charakter und die Zimmer waren sauber.

[Review 2]
Language: en
Domain: hotel
Similarity: 0.922
Text: The rooms were clean  hotel staff friendly .

[Review 3]
Language: de
Domain: hotel
Similarity: 0.917
Text: Die Zimmer waren sauber Hotelpersonal freundlich.

[Review 4]
Language: en
Domain: hotel
Similarity: 0.881
Text: Hotel has alot of charater and the rooms were clean .

[Review 5]
Language: de
Domain: hotel
Similarity: 0.877
Text: Die Einrichtung war wirklich modern und schön, das Hotel war sauber und die Betten waren komfortabel.

Return the result ONLY as valid JSON with this structure:

{
    "main_finding": "short summary of the main finding",
    "common_opinion": "what customers commonly think",
    "negative_signals": [
        "n

In [14]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


In [15]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

print("OpenAI client initialized")

OpenAI client initialized


In [16]:
def generate_rag_answer(query, context):

    prompt = f"""
You are a review intelligence assistant.

Analyze the user query using ONLY the retrieved customer reviews.

User query:
{query}

Retrieved customer reviews:
{context}

Return ONLY valid JSON with exactly this structure:

{{
    "main_finding": "main finding based on the retrieved reviews",
    "common_opinion": "common opinion explicitly supported by the reviews",
    "negative_signals": [],
    "key_aspects": [],
    "evidence": [
        {{
            "review": "short quote or paraphrase from a retrieved review",
            "supports": "what this evidence supports"
        }}
    ],
    "conclusion": "short evidence-based conclusion"
}}

Rules:
- Use ONLY the retrieved reviews as evidence.
- Do NOT use outside knowledge.
- Do NOT invent customer opinions.
- Do NOT claim that all customers think something unless the retrieved reviews support this.
- Distinguish between "the retrieved reviews indicate" and "all customers".
- If there are no negative signals, return [].
- Include 2-3 pieces of evidence.
- Keep the answer concise.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content

In [17]:
query = "The hotel room was very clean"

results = retrieve_reviews(
    query,
    top_k=5,
    domain="hotel"
)

context = build_context(results)

answer = generate_rag_answer(
    query,
    context
)

print(answer)

{
    "main_finding": "The hotel rooms are generally perceived as clean.",
    "common_opinion": "The cleanliness of the hotel rooms is consistently mentioned across reviews.",
    "negative_signals": [],
    "key_aspects": ["Room cleanliness", "Hotel overall condition"],
    "evidence": [
        {
            "review": "The rooms were clean  hotel staff friendly .",
            "supports": "Direct confirmation of room cleanliness"
        },
        {
            "review": "Die Zimmer waren sauber Hotelpersonal freundlich.",
            "supports": "German review explicitly stating room cleanliness"
        },
        {
            "review": "Hotel has alot of charater and the rooms were clean .",
            "supports": "Additional confirmation of room cleanliness"
        }
    ],
    "conclusion": "Multiple reviews confirm the hotel rooms are clean, supporting the user's observation."
}


In [18]:
answer_json = json.loads(answer)

display(answer_json)

{'main_finding': 'The hotel rooms are generally perceived as clean.',
 'common_opinion': 'The cleanliness of the hotel rooms is consistently mentioned across reviews.',
 'negative_signals': [],
 'key_aspects': ['Room cleanliness', 'Hotel overall condition'],
 'evidence': [{'review': 'The rooms were clean  hotel staff friendly .',
   'supports': 'Direct confirmation of room cleanliness'},
  {'review': 'Die Zimmer waren sauber Hotelpersonal freundlich.',
   'supports': 'German review explicitly stating room cleanliness'},
  {'review': 'Hotel has alot of charater and the rooms were clean .',
   'supports': 'Additional confirmation of room cleanliness'}],
 'conclusion': "Multiple reviews confirm the hotel rooms are clean, supporting the user's observation."}

In [19]:
def run_rag(
    query,
    top_k=5,
    language=None,
    domain=None
):
    # 1. Retrieve
    results = retrieve_reviews(
        query=query,
        top_k=top_k,
        language=language,
        domain=domain
    )

    # 2. Build context
    context = build_context(results)

    # 3. Generate answer
    answer = generate_rag_answer(
        query=query,
        context=context
    )

    return {
        "query": query,
        "retrieved_reviews": results,
        "context": context,
        "answer": answer
    }

In [20]:
test_queries = [
    {
        "query": "The hotel room was very clean",
        "language": "en",
        "domain": "hotel"
    },
    {
        "query": "Номер в отеле был очень чистым",
        "language": "ru",
        "domain": "hotel"
    },
    {
        "query": "Das Hotelzimmer war sehr sauber",
        "language": "de",
        "domain": "hotel"
    },
    {
        "query": "The laptop battery drains very quickly",
        "language": "en",
        "domain": "laptop"
    },
    {
        "query": "Батарея ноутбука быстро разряжается",
        "language": "ru",
        "domain": "laptop"
    },
    {
        "query": "Der Akku des Laptops hält nicht lange",
        "language": "de",
        "domain": "laptop"
    },
    {
        "query": "The food was delicious but the service was slow",
        "language": "en",
        "domain": "restaurant"
    },
    {
        "query": "Еда была вкусной, но обслуживание медленным",
        "language": "ru",
        "domain": "restaurant"
    },
    {
        "query": "Das Essen war lecker, aber der Service war langsam",
        "language": "de",
        "domain": "restaurant"
    }
]

In [ ]:
rag_outputs = []

for item in test_queries:

    print("=" * 100)
    print("QUERY:", item["query"])

    result = run_rag(
        query=item["query"],
        top_k=5,
        domain=item["domain"]
    )

    print("\nANSWER:")
    print(result["answer"])

    rag_outputs.append(result)


ANSWER:
{
    "main_finding": "The retrieved reviews mention battery life but do not directly address rapid battery drainage.",
    "common_opinion": "Positive feedback about battery life duration",
    "negative_signals": [],
    "key_aspects": ["battery life", "laptop weight"],
    "evidence": [
        {
            "review": "5 pound laptop with its nine hour battery life",
            "supports": "Indicates positive battery life duration"
        },
        {
            "review": "Akkulaufzeit: ...problemlos 8 Stunden oder mehr",
            "supports": "Supports positive battery life claims"
        }
    ],
    "conclusion": "The reviews indicate customers report long battery life, which contrasts with the user's experience of rapid drainage."
}
QUERY: Батарея ноутбука быстро разряжается


In [ ]:
def generate_llm_only(query):

    prompt = f"""
You are a review intelligence assistant.

Analyze the following user query.

User query:
{query}

Return ONLY valid JSON:

{{
    "main_finding": "short summary",
    "common_opinion": "common customer opinion",
    "negative_signals": [],
    "key_aspects": [],
    "conclusion": "short conclusion"
}}

Do not invent specific customer evidence.
Keep the answer concise.
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.message.content

In [ ]:
query = "The hotel room was very clean"

print("===== LLM ONLY =====")

llm_answer = generate_llm_only(query)

print(llm_answer)


print("\n===== RAG =====")

rag_result = run_rag(
    query=query,
    top_k=5,
    domain="hotel"
)

print(rag_result["answer"])

In [ ]:
query = "The hotel room was very clean"

rag_result = run_rag(
    query=query,
    top_k=5,
    domain="hotel"
)

print(rag_result["answer"])